In [9]:
import numpy as np
import pandas as pd

In [3]:
# Global Variables
FILE_PATH: str = "https://raw.githubusercontent.com/vinayak-ensemble/Dataset-TV-Shows-OTT/refs/heads/main/tv-shows.csv" 
OTT_DF: pd.DataFrame = pd.read_csv(FILE_PATH)

## 1. Pre-processing

In [4]:
# Converting the date_added to datetime format:
OTT_DF['date_added'] = pd.to_datetime(OTT_DF['date_added'].str.strip(), format='%B %d, %Y')

In [5]:
OTT_DF.columns

Index(['id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'platform'],
      dtype='str')

In [6]:
# Handle missing values:

# -- High missing value columns --

# Treatment: 
# - need to fill in `Unknown` for the missing values in these cases.
# - since director has a 30% missing value we need to check if it's fit to be modeled on. 

treatment_cols = ['director', 'cast', 'country']
OTT_DF[treatment_cols] = OTT_DF[treatment_cols].fillna('Unknown')
OTT_DF[treatment_cols].isnull().sum()

director    0
cast        0
country     0
dtype: int64

In [7]:
# Handle missing values:

# -- Low missing value columns --

# Treatment: 
# - need to fill in with the mean of the numeric columns.
# - need to fill in with the mode of the string columns. 

mean_treat_cols = ['date_added' ,'release_year']
mode_treat_simp_cols = ['rating']
mode_tread_grp_cols = ['duration']

# fillna with mean of the columns
for col in mean_treat_cols:
    OTT_DF[col] = OTT_DF[col].fillna(OTT_DF[col].mean())

# fillna with the mode of the column
OTT_DF[mode_treat_simp_cols] = OTT_DF[mode_treat_simp_cols].fillna(OTT_DF[mode_treat_simp_cols].mode().values[0][0])

# fillna with the mode of the column by type
duration_mode_by_type = (
    OTT_DF.groupby('type')['duration']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

OTT_DF['duration'] = OTT_DF['duration'].fillna(
    OTT_DF['type'].map(duration_mode_by_type)
)

OTT_DF[mean_treat_cols + mode_treat_simp_cols + mode_tread_grp_cols].isnull().sum()

date_added      0
release_year    0
rating          0
duration        0
dtype: int64

## 2. Data Transformation

In [10]:
# Transform the duration column into 2 seperate columns based on type.

# extract numeric value
OTT_DF['duration_num'] = OTT_DF['duration'].str.extract(r'(\d+)').astype(float)

# create separate features
OTT_DF['duration_movies'] = np.where(
    OTT_DF['type'] == 'Movie',
    OTT_DF['duration_num'],
    np.nan
)

OTT_DF['duration_tv_series'] = np.where(
    OTT_DF['type'] == 'TV Show',
    OTT_DF['duration_num'],
    np.nan
)

# optional: fill missing values with 0
OTT_DF[['duration_movies', 'duration_tv_series']] = (
    OTT_DF[['duration_movies', 'duration_tv_series']]
    .fillna(0)
)

# drop old columns
OTT_DF.drop(columns=['duration', 'duration_num'], inplace=True)
OTT_DF[['duration_movies', 'duration_tv_series']].head()

,duration_movies,duration_tv_series
0,61.0,0.0
1,123.0,0.0
2,0.0,1.0
3,0.0,1.0
4,63.0,0.0


In [16]:
# Transform the datato create meaningful bins

# using quartile bins to have more or less euqal number of values in each bin
OTT_DF['date_added_year_bin'] = pd.qcut(
    OTT_DF['date_added'].dt.year,
    q=6,
    labels=False,
    duplicates='drop'
)

OTT_DF['release_year_bin'] = pd.qcut(
    OTT_DF['release_year'],
    q=6,
    labels=False,
    duplicates='drop'
)

OTT_DF.drop(
    columns=['date_added', 'release_year'],
    inplace=True
)

In [19]:
print(OTT_DF['date_added_year_bin'].value_counts(), end='\n\n')
print(OTT_DF['release_year_bin'].value_counts())

date_added_year_bin
1    4055
2    2030
3    1678
0    1575
Name: count, dtype: int64

release_year_bin
2    2414
4    1997
0    1678
1    1498
3    1086
5     665
Name: count, dtype: int64


In [ ]:
# TODO:
# TF-IDF on the categorical columns.
# Check which are the columns that would help classify the categories best and then iterate through it.
# Try hyper-Parameter tuning
# Cross Validation
# Active learning (if possible)